# 12 · Descarga para la ampliación de partidos (experimento)

Descarga **NASA POWER** y **NDVI MODIS** para los 12 partidos nuevos de `PARTIDOS_AMPLIACION` (`src/config.py`), replicando el método de los notebooks 01 y 02:

- **Clima:** un punto por partido (centroide GAUL 2015, fijo en `config.py`), mismos parámetros y período que el notebook 01.
- **NDVI:** media de MOD13Q1 sobre el polígono FAO GAUL 2015 nivel 2 (Argentina + Buenos Aires + nombre), sin máscara de cultivo, igual que el notebook 02.

Salida en `data/raw/ampliacion/` (no toca los CSV de los 5 core). Correr las celdas en orden, desde la carpeta del proyecto.

In [1]:
# Paths del proyecto: centralizados en src/config.py
import sys
from pathlib import Path

_raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "config.py").exists()), None)
if _raiz is None:
    raise FileNotFoundError("No se encontró src/config.py: abrí el notebook desde la carpeta del proyecto")
if str(_raiz / "src") not in sys.path:
    sys.path.insert(0, str(_raiz / "src"))
from config import PROJECT_ROOT, DATA_RAW, DATA_PROCESSED, OUTPUTS, PARTIDOS_AMPLIACION, DATA_RAW_AMPLIACION

print(f"{len(PARTIDOS_AMPLIACION)} partidos nuevos:", list(PARTIDOS_AMPLIACION))
assert len(PARTIDOS_AMPLIACION) == 12

12 partidos nuevos: ['Pergamino', 'Colón', 'Rojas', 'Salto', 'Chacabuco', 'Bragado', '9 de Julio', 'General Viamonte', 'General Villegas', 'Pehuajó', 'Carlos Tejedor', 'Trenque Lauquen']


## 1. NASA POWER (mismo método que el notebook 01)

In [2]:
import requests
import pandas as pd
import numpy as np
import time

PARAMS = "T2M,T2M_MAX,T2M_MIN,PRECTOTCORR,ALLSKY_SFC_SW_DWN,RH2M,T2MDEW"
BASE_URL = "https://power.larc.nasa.gov/api/temporal/daily/point"

OUTPUT_DIR_NASA = DATA_RAW_AMPLIACION / "nasa_power"
OUTPUT_DIR_NASA.mkdir(parents=True, exist_ok=True)
OUTPUT_NASA = OUTPUT_DIR_NASA / "nasa_power_ampliacion_1999_2025.csv"

all_data = []

for partido, coords in PARTIDOS_AMPLIACION.items():
    print(f"Descargando {partido}...", end=" ", flush=True)

    url = (
        f"{BASE_URL}?"
        f"parameters={PARAMS}"
        f"&community=AG"
        f"&longitude={coords['lon']}"
        f"&latitude={coords['lat']}"
        f"&start=19990901"
        f"&end=20250430"
        f"&format=JSON"
    )

    resp = requests.get(url, timeout=120)
    resp.raise_for_status()
    data = resp.json()

    df = pd.DataFrame(data["properties"]["parameter"])
    df.index.name = "fecha"
    df = df.reset_index().rename(columns={"index": "fecha"})
    df["partido"] = partido
    df["lat"] = coords["lat"]
    df["lon"] = coords["lon"]
    all_data.append(df)
    print(f"OK — {len(df)} días")
    time.sleep(2)

# Unir y limpiar
df_clima = pd.concat(all_data, ignore_index=True)
df_clima["fecha"] = pd.to_datetime(df_clima["fecha"], format="%Y%m%d")

# -999 = missing de NASA POWER → NaN
numeric_cols = ["T2M","T2M_MAX","T2M_MIN","PRECTOTCORR","ALLSKY_SFC_SW_DWN","RH2M","T2MDEW"]
for col in numeric_cols:
    df_clima[col] = df_clima[col].replace(-999, np.nan)

df_clima.to_csv(OUTPUT_NASA, index=False)

print(f"\nListo! Shape: {df_clima.shape}")
print(f"Período: {df_clima['fecha'].min().date()} a {df_clima['fecha'].max().date()}")
print(f"Guardado en: {OUTPUT_NASA}")
print(df_clima.groupby('partido').size())
print("Nulos:\n", df_clima[numeric_cols].isnull().sum())

Descargando Pergamino... OK — 9374 días
Descargando Colón... OK — 9374 días
Descargando Rojas... OK — 9374 días
Descargando Salto... OK — 9374 días
Descargando Chacabuco... OK — 9374 días
Descargando Bragado... OK — 9374 días
Descargando 9 de Julio... OK — 9374 días
Descargando General Viamonte... OK — 9374 días
Descargando General Villegas... OK — 9374 días
Descargando Pehuajó... OK — 9374 días
Descargando Carlos Tejedor... OK — 9374 días
Descargando Trenque Lauquen... OK — 9374 días

Listo! Shape: (112488, 11)
Período: 1999-09-01 a 2025-04-30
Guardado en: C:\Users\agust\Documents\rinde-soja-github\data\raw\ampliacion\nasa_power\nasa_power_ampliacion_1999_2025.csv
partido
9 de Julio          9374
Bragado             9374
Carlos Tejedor      9374
Chacabuco           9374
Colón               9374
General Viamonte    9374
General Villegas    9374
Pehuajó             9374
Pergamino           9374
Rojas               9374
Salto               9374
Trenque Lauquen     9374
dtype: int64
Nulos

## 2. NDVI MODIS vía GEE (mismo método que el notebook 02)

La autenticación solo se pide si `ee.Initialize` falla (los notebooks 01 y 02 quedan como están).

In [3]:
import ee
try:
    ee.Initialize(project='rinde-soja')
except Exception:
    ee.Authenticate()
    ee.Initialize(project='rinde-soja')
print("Earth Engine listo!")

Earth Engine listo!


In [4]:
# Nombre en FAO GAUL (sin tildes) -> nombre usado en el proyecto
GAUL_A_PROYECTO = {d["gaul"]: nombre for nombre, d in PARTIDOS_AMPLIACION.items()}
OUTPUT_NDVI = DATA_RAW_AMPLIACION / "ndvi" / "ndvi_modis_ampliacion_2000_2025.csv"

# Polígonos: Argentina + provincia de Buenos Aires + nombre exacto
gaul_ba = (ee.FeatureCollection("FAO/GAUL/2015/level2")
           .filter(ee.Filter.eq("ADM0_NAME", "Argentina"))
           .filter(ee.Filter.eq("ADM1_NAME", "Buenos Aires")))

# Verificación POR PARTIDO: cada nombre debe devolver EXACTAMENTE 1 polígono
for gaul_name in GAUL_A_PROYECTO:
    n = gaul_ba.filter(ee.Filter.eq("ADM2_NAME", gaul_name)).size().getInfo()
    assert n == 1, f"'{gaul_name}': se esperaba EXACTAMENTE 1 polígono y hay {n}"

partidos_fc = gaul_ba.filter(ee.Filter.inList("ADM2_NAME", list(GAUL_A_PROYECTO.keys())))
n = partidos_fc.size().getInfo()
assert n == 12, f"Se esperaban 12 polígonos y se encontraron {n}"
print("Polígonos OK (1 por partido):", sorted(partidos_fc.aggregate_array("ADM2_NAME").getInfo()))

Polígonos OK (1 por partido): ['9 de Julio', 'Bragado', 'Carlos Tejedor', 'Chacabuco', 'Colon', 'General Viamonte', 'General Villegas', 'Pehuajo', 'Pergamino', 'Rojas', 'Salto', 'Trenque Lauquen']


In [5]:
# MOD13Q1: NDVI cada 16 días, resolución 250 m, desde 2000
modis = ee.ImageCollection("MODIS/061/MOD13Q1").filterDate("2000-01-01", "2025-04-30").select("NDVI")

def extract_ndvi(image):
    fecha = image.date().format("YYYY-MM-dd")
    medias = image.reduceRegions(collection=partidos_fc, reducer=ee.Reducer.mean(), scale=250)
    return medias.map(lambda f: f.set("fecha", fecha))

# Año por año para no saturar la memoria de GEE
print("Extrayendo NDVI por partido (esto tarda varios minutos)...")
registros = []
for year in range(2000, 2026):
    fin = f"{year}-12-31" if year < 2025 else "2025-04-30"
    try:
        features = modis.filterDate(f"{year}-01-01", fin).map(extract_ndvi).flatten().getInfo()["features"]
        for f in features:
            p = f["properties"]
            registros.append({"fecha": p["fecha"], "partido_gaul": p["ADM2_NAME"], "ndvi_raw": p.get("mean")})
        print(f"  {year}: {len(features)} registros — OK")
    except Exception as e:
        print(f"  {year}: ERROR — {e}")

df_ndvi = pd.DataFrame(registros)
df_ndvi["fecha"] = pd.to_datetime(df_ndvi["fecha"])
df_ndvi["ndvi"] = df_ndvi["ndvi_raw"] * 0.0001          # factor de escala de MOD13Q1
df_ndvi["partido"] = df_ndvi["partido_gaul"].map(GAUL_A_PROYECTO)
df_ndvi = df_ndvi.sort_values(["partido", "fecha"]).reset_index(drop=True)

OUTPUT_NDVI.parent.mkdir(parents=True, exist_ok=True)
df_ndvi.to_csv(OUTPUT_NDVI, index=False)

print(f"\n{'='*50}")
print(f"Guardado en: {OUTPUT_NDVI}")
print(f"Shape: {df_ndvi.shape}")
print(f"Período: {df_ndvi['fecha'].min().date()} a {df_ndvi['fecha'].max().date()}")
print(f"Partidos: {df_ndvi['partido'].unique().tolist()}")
print(f"NDVI rango: {df_ndvi['ndvi'].min():.3f} a {df_ndvi['ndvi'].max():.3f}")

Extrayendo NDVI por partido (esto tarda varios minutos)...
  2000: 240 registros — OK
  2001: 276 registros — OK
  2002: 276 registros — OK
  2003: 276 registros — OK
  2004: 276 registros — OK
  2005: 276 registros — OK
  2006: 276 registros — OK
  2007: 276 registros — OK
  2008: 276 registros — OK
  2009: 276 registros — OK
  2010: 276 registros — OK
  2011: 276 registros — OK
  2012: 276 registros — OK
  2013: 276 registros — OK
  2014: 276 registros — OK
  2015: 276 registros — OK
  2016: 276 registros — OK
  2017: 276 registros — OK
  2018: 276 registros — OK
  2019: 276 registros — OK
  2020: 276 registros — OK
  2021: 276 registros — OK
  2022: 276 registros — OK
  2023: 276 registros — OK
  2024: 276 registros — OK
  2025: 96 registros — OK

Guardado en: C:\Users\agust\Documents\rinde-soja-github\data\raw\ampliacion\ndvi\ndvi_modis_ampliacion_2000_2025.csv
Shape: (6960, 5)
Período: 2000-02-18 a 2025-04-23
Partidos: ['9 de Julio', 'Bragado', 'Carlos Tejedor', 'Chacabuco', 'Coló

In [6]:
# Verificación: mismos registros por partido (580 en los core), sin duplicados ni nulos
print(df_ndvi.groupby("partido").size())
print("Duplicados (fecha, partido):", df_ndvi.duplicated(subset=["fecha", "partido"]).sum())
print("NDVI nulos:", df_ndvi["ndvi"].isna().sum())
print("Partidos sin mapear:", df_ndvi["partido"].isna().sum())

partido
9 de Julio          580
Bragado             580
Carlos Tejedor      580
Chacabuco           580
Colón               580
General Viamonte    580
General Villegas    580
Pehuajó             580
Pergamino           580
Rojas               580
Salto               580
Trenque Lauquen     580
dtype: int64
Duplicados (fecha, partido): 0
NDVI nulos: 0
Partidos sin mapear: 0
